<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 18 · Structural Credit Models

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch18_structural_credit_models.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

This notebook computes structural credit risk with the Merton model. You will

- read default as an asset-value event: equity is a call option on firm assets with strike equal to debt face value,
- compute distance to default from log leverage, asset volatility, the risk-free rate, and the horizon,
- translate distance to default into a model-implied default probability through the standard normal CDF,
- run leverage and volatility scenarios on the frozen inputs and see how quickly the result moves, and
- judge how far a structural number can be trusted, given that asset value and asset volatility are latent.

The outputs are transparent model diagnostics, not underwriting decisions.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The cell below runs the Chapter 18 script. It loads the frozen Merton inputs, then evaluates distance to default and the model-implied default probability for four scenarios: base, higher leverage, higher volatility, and stress. Distance to default is measured in asset-value standard deviations over the one-year horizon, and the default probability is the normal CDF evaluated at minus that distance.

In [ ]:
SCRIPT = ROOT / "code" / "ch18_structural_credit_models.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

- Verify the base case: ln(120/100) plus (0.04 minus half of 0.25 squared) gives a numerator of 0.191072; divided by 0.25, distance to default is 0.764286 and the default probability is 0.222348.
- Check the leverage effect: cutting asset value from 120 to 105 at unchanged volatility lowers distance to default from 0.764286 to 0.230161 and raises the default probability from 22.2% to 40.9%.
- Separate the volatility effect: raising volatility from 0.25 to 0.35 at unchanged leverage gives 0.460204 and 32.3%, because the asset-value left tail gets fatter.
- Note the stress case turns distance to default negative at -0.100000, so expected assets end below debt face value and the default probability exceeds 50%.
- Record the latent inputs, asset value, asset volatility, debt face value, and horizon, with any reported number; they fully determine the result.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch18_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Merton Distance to Default From Leverage and Volatility

Distance to default measures how many asset-value standard deviations separate the firm from its debt face value at the horizon. The numerator combines log leverage, ln of asset value over debt face value, with the drift term (risk-free rate minus half the asset variance) times the horizon; the denominator is asset volatility times the square root of the horizon. The model-implied default probability is the standard normal CDF at minus distance to default. The three scenarios below isolate leverage and volatility one at a time, then together.

In [ ]:
from math import erf, log, sqrt
import pandas as pd

def normal_cdf(x):  # standard normal CDF
    return 0.5 * (1 + erf(x / sqrt(2)))

def dd(v, debt, vol, rate, horizon):
    drift = (rate - 0.5 * vol ** 2) * horizon
    return (log(v / debt) + drift) / (vol * sqrt(horizon))

rows = [  # illustrative scenarios: asset value, asset vol
    ("base", 120.0, 0.25),
    ("higher leverage", 105.0, 0.25),
    ("stress", 100.0, 0.40),
]
table = pd.DataFrame(rows, columns=["scenario", "asset_value", "vol"])
table["distance_to_default"] = [
    dd(r[1], 100.0, r[2], 0.04, 1.0) for r in rows
]
table["default_probability"] = [
    normal_cdf(-d) for d in table.distance_to_default
]
table.round(4)

### What to observe

Base leverage, assets of 120 against 100 of debt at 25% volatility, gives a distance to default of 0.7643 and a default probability of 22.2%. Lowering asset value to 105 at the same volatility cuts the distance to 0.2302 and lifts the probability to 40.9%. The stress case, assets of 100 at 40% volatility, turns the distance negative at -0.1000 with a 54.0% probability.

## Exercises

1. **Distance by hand.** Recompute the base case from ln(120/100), a 4% risk-free rate, 25% volatility, and a one-year horizon. *Hint:* the numerator is 0.182322 plus 0.008750.
2. **Volatility scenario.** Compute distance to default and default probability for assets of 120 and 35% volatility, and match the script's higher-volatility row.
3. **Direction checks.** Explain in one sentence each why higher leverage and higher volatility both raise the default probability.
4. **Horizon effect.** Recompute the base case with a two-year horizon and interpret the change in the default probability.
5. **Latent inputs.** List two reasons a Merton default probability is hard to calibrate for a real borrower. *Hint:* asset value and volatility are unobserved, and default can occur before maturity.
6. **Structural versus reduced form.** State in two sentences when you would use a structural distance to default and when a market-implied hazard rate.

For more detail, see Chapter 18 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">